## 📊 3. Fase EXPLORE: Análise Exploratória Univariada

Exploração individual das *features* extraídas das imagens e consolidadas na ABT (`data/processed/abt_sanidade_vegetal.csv`)[cite: 1]. O objetivo é mapear distribuições, detectar *outliers* estruturais e validar o balanceamento da variável alvo antes da etapa de modelagem (SVM).

In [1]:
# 3.1 Setup do Motor de Análise Visual (Plotly)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

class AdvancedVisionEDA:
    """
    Pipeline interativo de Análise Univariada.
    Gera dashboards dinâmicos para identificação de padrões e anomalias nas features de imagem.
    """
    def __init__(self, df: pd.DataFrame, target_col: str):
        self.df = df
        self.target_col = target_col
        self.num_cols = self.df.select_dtypes(include=['float64', 'int64']).columns.tolist()
        
        # Remove as colunas de target da análise contínua para evitar distorções
        cols_to_remove = [self.target_col, 'target_binary', 'target_multiclass']
        for col in cols_to_remove:
            if col in self.num_cols:
                self.num_cols.remove(col)

    def plot_target_distribution(self):
        """Avalia o balanceamento das classes de sanidade."""
        class_counts = self.df[self.target_col].value_counts().reset_index()
        class_counts.columns = [self.target_col, 'Contagem']
        
        fig = px.bar(class_counts, x=self.target_col, y='Contagem', 
                     text='Contagem', color='Contagem', color_continuous_scale='Teal',
                     title="Balanceamento das Classes de Sanidade Vegetal",
                     template="plotly_dark")
        fig.update_traces(textposition='outside')
        fig.show()

    def plot_continuous_features(self):
        """Gera Histograma + Boxplot para variáveis contínuas."""
        for col in self.num_cols:
            fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                                vertical_spacing=0.05, row_heights=[0.2, 0.8])
            
            fig.add_trace(go.Box(x=self.df[col], name="Boxplot", marker_color="#00CC96"), row=1, col=1)
            fig.add_trace(go.Histogram(x=self.df[col], name="Distribuição", marker_color="#636EFA", nbinsx=50), row=2, col=1)
            
            fig.update_layout(title_text=f"Distribuição e Outliers: `{col}`", showlegend=False, 
                              template="plotly_dark", height=450)
            fig.show()

    def generate_descriptive_stats(self):
        """Exibe as estatísticas descritivas, skewness e kurtosis."""
        stats = self.df[self.num_cols].describe().T
        stats['skewness'] = self.df[self.num_cols].skew()
        stats['kurtosis'] = self.df[self.num_cols].kurt()
        display(stats.round(3).style.background_gradient(cmap='viridis'))

# 3.2 Execução do Pipeline Univariado na ABT
abt_path = '../data/processed/abt_sanidade_vegetal.csv'
df_features = pd.read_csv(abt_path)

# A coluna correta do seu dataset é 'class_label'
eda_pipeline = AdvancedVisionEDA(df=df_features, target_col='class_label')
eda_pipeline.plot_target_distribution()
eda_pipeline.plot_continuous_features()
eda_pipeline.generate_descriptive_stats()

,count,mean,std,min,25%,50%,75%,max,skewness,kurtosis
width,6571.000000,1240.547000,1645.311000,640.000000,640.000000,640.000000,640.000000,6016.000000,2.524000,4.450000
height,6571.000000,1046.936000,1046.661000,640.000000,640.000000,640.000000,640.000000,6005.000000,2.402000,4.012000
size_kb,6571.000000,843.958000,2287.298000,17.300000,44.400000,60.610000,81.485000,14517.640000,2.744000,6.111000
mean_hue,6571.000000,48.989000,18.129000,0.470000,35.400000,48.500000,62.640000,101.750000,0.040000,-0.865000
std_saturation,6571.000000,0.198000,0.064000,0.029000,0.145000,0.197000,0.245000,0.394000,0.180000,-0.677000
exg_index,6571.000000,21.988000,12.646000,-7.380000,12.465000,20.260000,29.465000,61.780000,0.485000,-0.416000
exr_index,6571.000000,4.661000,13.690000,-31.630000,-4.860000,4.460000,13.410000,49.750000,0.182000,-0.447000
glcm_contrast,6571.000000,25.516000,9.315000,3.870000,18.780000,25.580000,31.565000,62.640000,0.263000,-0.213000
glcm_homogeneity,6571.000000,0.694000,0.116000,0.347000,0.612000,0.684000,0.764000,0.982000,0.220000,-0.588000
laplacian_var,6571.000000,156.277000,54.499000,21.590000,123.450000,145.850000,169.570000,450.710000,1.606000,3.054000


## 🎯 Conclusões da Análise Univariada

✅ **Principais Padrões e Observações (Checklist):**

1. **Distribuição da Variável Alvo:** Identificamos um **desbalanceamento** evidente entre as instâncias. Enquanto as classes *Red Rot* e *Mosaic* lideram com mais de 1.200 amostras, *Leaf Scald* (439) e *Grassy Shoot* (206) são minoritárias, o que exigirá técnicas de tratamento de classes desbalanceadas na modelagem.
2. **Variáveis Contínuas (Padrões Visuais):** Descritores como Excesso de Verde (`exg_index`) apresentam distribuição **aproximadamente normal** (formato de sino e *skewness* baixo de 0.48), o que corrobora a hipótese de separabilidade matemática clara entre tecido saudável e necrosado.
3. **Concentrações e Outliers:** A análise via boxplots revelou **presença significativa** de *outliers* do lado direito nas métricas de textura (`glcm_contrast` e `laplacian_var`), além de anomalias severas na dimensão original das imagens (`width` e `height`), confirmando a urgência de uma normalização robusta (como *StandardScaler*) no pipeline da Sprint 2.